In [0]:
import pyspark.sql.functions as F
from datetime import datetime

###What is Z-Order?

**Z-Order (ZORDER BY)** is a data layout optimization in Delta Lake that co-locates related data in the same set of files, enabling Delta to skip large amounts of irrelevant data during queries via **data skipping**.

---

**How it works**
When you run `OPTIMIZE ... ZORDER BY (col1, col2)`, Delta Lake rewrites the table's data files using a **Z-order space-filling curve** — a multi-dimensional indexing technique. It maps multiple columns into a single linear ordering such that values that are close together in N-dimensional space remain close together in the file layout. The result: rows with similar values in the Z-ordered columns end up in the same files.

Delta tracks **min/max statistics** per file per column. At query time, if a `WHERE` clause filter can be evaluated against those stats, entire files are skipped without being read — this is called **file skipping**.

---
**When to use Z-Order**

Z-Order is effective when:
* Queries frequently filter on specific columns (e.g., `WHERE user_id = ...` or `WHERE event_date = ...`)
* The column has **high cardinality** (many distinct values like user_id, transaction_id)
* You have large tables where file skipping yields meaningful I/O savings
* You can't or don't want to use partitioning (Z-Order works inside partitions too)

---
**Z-Order vs Partitioning vs Liquid Clustering**

| Feature | Partitioning | Z-Order | Liquid Clustering |
|---|---|---|---|
| Works on | Low-cardinality cols | Any column | Any column (auto) |
| Maintenance | Manual | Manual (`OPTIMIZE`) | Automatic (incremental) |
| Multi-column | No | Yes (up to ~4 cols) | Yes |
| Recommended for | Legacy/external tables | UC tables w/ manual ops | UC managed tables (preferred) |

---
**Key limitations**

* Z-Order must be re-run manually via `OPTIMIZE` — it does not apply to newly added files automatically
* Effectiveness degrades with too many Z-ordered columns (diminishing returns beyond 3-4)
* It rewrites data files, which can be expensive on large tables
* **Liquid Clustering** (with `CLUSTER BY`) is the modern replacement — it supports incremental clustering and is the recommended approach for Unity Catalog managed tables

---
**Basic syntax**

```sql
OPTIMIZE schema.table_name
ZORDER BY (column1, column2);
```


**Summary**
- Z-Order is best understood as: *sort the data so that what you query together lives together*, giving Delta's file-skipping the best chance to skip irrelevant files entirely.
- In Simple words Z-Order logically sorts and repartitions/rewrites similar data in same file. 

In [0]:
df = spark.read.parquet(
    "abfss://dalta-lake-lab-sacc-container@daltalakelabstorageacc.dfs.core.windows.net/invoices/invoices_201_99457.parquet"
)
print(df.count())

- We are doing union on same data below to mimic a seen where most file contains a overlaping row/same data between all files which will cause performance degradation.
- Then will apply Z-order to validate the performance improvement.

In [0]:
df_union = df
expected_rows = 10000000 #1cr

while df_union.count() <= expected_rows:
    df_union = df_union.union(df_union)

print("df_union rows count :- ", df_union.count())

In [0]:
df_union.write.mode("overwrite").saveAsTable("delta_catalog.delta_db.zorder_tbl_ex1")

In [0]:
%sql
-- UC-managed tables block direct dbutils.fs.ls() on __unitystorage paths.
-- Use _metadata.file_path to list physical files for the partition instead.
SELECT count(DISTINCT _metadata.file_path) AS file_count
FROM delta_catalog.delta_db.zorder_tbl_ex1

In [0]:
import time

start = time.time()

result = spark.sql("""
    WITH base AS (
        SELECT
            customer_id,
            category,
            gender,
            shopping_mall,
            quantity * price   AS line_total,
            quantity,
            price,
            invoice_date
        FROM delta_catalog.delta_db.zorder_tbl_ex1 VERSION AS OF 0
        -- Narrow, highly selective filter -> forces file skipping to matter
        WHERE customer_id BETWEEN 5000 AND 8000
    ),
    agg AS (
        SELECT
            category,
            gender,
            shopping_mall,
            COUNT(*)                         AS txn_count,
            COUNT(DISTINCT customer_id)      AS unique_customers,
            SUM(line_total)                  AS total_sales,
            AVG(line_total)                  AS avg_sale,
            MAX(line_total)                  AS max_sale,
            MIN(line_total)                  AS min_sale,
            SUM(quantity)                    AS total_qty,
            SUM(line_total) / SUM(quantity)  AS revenue_per_unit
        FROM base
        GROUP BY category, gender, shopping_mall
    )
    SELECT
        category,
        gender,
        shopping_mall,
        txn_count,
        unique_customers,
        ROUND(total_sales, 2)                                               AS total_sales,
        ROUND(avg_sale, 2)                                                  AS avg_sale,
        ROUND(max_sale, 2)                                                  AS max_sale,
        ROUND(min_sale, 2)                                                  AS min_sale,
        total_qty,
        ROUND(revenue_per_unit, 2)                                          AS revenue_per_unit,
        RANK() OVER (PARTITION BY category ORDER BY total_sales DESC)       AS rank_in_category,
        ROUND(
            total_sales * 100.0 / SUM(total_sales) OVER (PARTITION BY category),
            2
        )                                                                   AS pct_of_category
    FROM agg
    ORDER BY category, rank_in_category
""")
# display() triggers actual execution
display(result)

end = time.time()
print(f"\n>>> WITHOUT Z-Order (VERSION 0): {end - start:.2f}s")

In [0]:
%sql
OPTIMIZE delta_catalog.delta_db.zorder_tbl_ex1
ZORDER BY customer_id; --In ZORDER By we put columns that we are using on our queries to filter/prune the table.

In [0]:
# from delta.tables import DeltaTable

# delta_table = DeltaTable.forName(spark, "delta_catalog.delta_db.zorder_tbl_ex1")
# delta_table.optimize().executeZOrderBy("customer_id")

In [0]:
%sql
-- UC-managed tables block direct dbutils.fs.ls() on __unitystorage paths.
-- Use _metadata.file_path to list physical files for the partition instead.
SELECT count(DISTINCT _metadata.file_path) AS file_count
FROM delta_catalog.delta_db.zorder_tbl_ex1

- So the Z-order tumstoned all old 1024 files and created 15 new files with colocated data **(when input was 10cr)**.
- So the Z-order tumstoned all old 128 files and created 1 new files with colocated data **(when input was 1cr)**.
- Now lets test same query.

In [0]:
import time

start = time.time()

result = spark.sql("""
    WITH base AS (
        SELECT
            customer_id,
            category,
            gender,
            shopping_mall,
            quantity * price   AS line_total,
            quantity,
            price,
            invoice_date
        FROM delta_catalog.delta_db.zorder_tbl_ex1
        -- Same narrow filter on Z-ordered column -> Delta skips most files
        WHERE customer_id BETWEEN 5000 AND 8000
    ),
    agg AS (
        SELECT
            category,
            gender,
            shopping_mall,
            COUNT(*)                         AS txn_count,
            COUNT(DISTINCT customer_id)      AS unique_customers,
            SUM(line_total)                  AS total_sales,
            AVG(line_total)                  AS avg_sale,
            MAX(line_total)                  AS max_sale,
            MIN(line_total)                  AS min_sale,
            SUM(quantity)                    AS total_qty,
            SUM(line_total) / SUM(quantity)  AS revenue_per_unit
        FROM base
        GROUP BY category, gender, shopping_mall
    )
    SELECT
        category,
        gender,
        shopping_mall,
        txn_count,
        unique_customers,
        ROUND(total_sales, 2)                                               AS total_sales,
        ROUND(avg_sale, 2)                                                  AS avg_sale,
        ROUND(max_sale, 2)                                                  AS max_sale,
        ROUND(min_sale, 2)                                                  AS min_sale,
        total_qty,
        ROUND(revenue_per_unit, 2)                                          AS revenue_per_unit,
        RANK() OVER (PARTITION BY category ORDER BY total_sales DESC)       AS rank_in_category,
        ROUND(
            total_sales * 100.0 / SUM(total_sales) OVER (PARTITION BY category),
            2
        )                                                                   AS pct_of_category
    FROM agg
    ORDER BY category, rank_in_category
""") 
# display() triggers actual execution
display(result)

end = time.time()
print(f"\n>>> WITH Z-Order (latest version): {end - start:.2f}s")

In [0]:
%sql
DESCRIBE HISTORY delta_catalog.delta_db.zorder_tbl_ex1;

### Z-order With Hive Style Partition

**Hive-style partitioning** is a method of organizing data in a directory structure where each partition column and its value form a subfolder, such as `invoice_date=2021-01-01`. This approach allows data to be physically separated based on partition column values, making queries on those columns more efficient. For example, a table partitioned by `invoice_date` will have folders like `invoice_date=2021-01-01`, `invoice_date=2021-01-02`, etc., each containing only the data for that date. Hive-style partitioning is widely used in big data systems like Apache Hive and Spark, enabling fast filtering and pruning of data during query execution. While Delta Lake supports partitioning, Hive-style partitioning refers specifically to this folder-based layout, which is not part of the Delta Lake protocol but is commonly used with Parquet and other file formats.

In [0]:
df.write.mode("overwrite").partitionBy("invoice_date").saveAsTable("delta_catalog.delta_db.zorder_tbl_ex2")

In [0]:
# spark.sql("""
#     ALTER TABLE delta_catalog.delta_db.zorder_tbl_ex2
#     SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 0 hours')
# """)

# spark.sql("""
#     VACUUM delta_catalog.delta_db.zorder_tbl_ex2
# """)

# 
for m in dir(F): 
    if "date" in m : print(m)

- In Case Of Hive Style Partition: i.e Data is distributed in among column example invoice_date folders/partitions
- Then we ZORDER BY: customer_id
- For each of those invoice_date

In [0]:
df_new_partiton = df.filter(F.col("invoice_date") == "2021-01-01").withColumn("invoice_date", F.lit(F.current_date()))
display(df_new_partiton.limit(5))
print(df_new_partiton.select("invoice_date").distinct().count())

In [0]:
df_new_partiton.write.mode("append").partitionBy("invoice_date").saveAsTable("delta_catalog.delta_db.zorder_tbl_ex2")

In [0]:
%sql
SELECT
  min(invoice_date),
  max(invoice_date)
FROM
  delta_catalog.delta_db.zorder_tbl_ex2;

In [0]:
%sql
OPTIMIZE
  delta_catalog.delta_db.zorder_tbl_ex2
WHERE
  invoice_date = '2026-06-26' --In Where we put whatever partition column we used.
ZORDER BY customer_id; --In ZORDER By we put columns that we are using on our queries to filter/prune the table.

In [0]:
%sql
DESCRIBE HISTORY delta_catalog.delta_db.zorder_tbl_ex2;

#### Conlusion On data optimization method *Hive-style partitioning* and *Z-Ordering*

*   **Hive-style Partitioning**: This organizes data by placing records into separate folders based on the value of a specific column (e.g., date). While this effectively skips irrelevant data during queries, it relies on static directory structures.
*   **Z-Ordering**: This is a space-filling curve technique that maps multi-dimensional data into a single dimension while preserving **locality**. By placing similar data points close together in files, it enables the engine to "skip" more data files, significantly reducing the amount of data sent over the wire.
*   **The Shared Limitation**: The instructor emphasizes that both approaches are **inflexible**. Because they require you to decide your partitioning and Z-Order columns **upfront** (at the time of table creation or during periodic maintenance), they cannot easily adapt to changing query patterns or uneven data growth (data skew) after the data has been laid out.
---

### What is Liquid Clustering(Advanced Optimization Technique) in detail?

Liquid Clustering is the modern replacement for both Hive-style partitioning and Z-Order in Delta Lake on Unity Catalog managed tables. Here's a detailed breakdown:

---
**What it is**
- Liquid Clustering uses a flexible, file-level clustering strategy based on a **space-filling curve** (similar to Z-Order internally), but it is **incremental** and **adaptive** — it doesn't require rewriting the entire table when you run `OPTIMIZE`.

---
**How it differs from Z-Order / Partitioning**

| | Hive Partitioning | Z-Order | Liquid Clustering |
|---|---|---|---|
| Column choice | At table creation | Any time, but full rewrite | At creation, changeable later |
| Rewrite scope | Full partition | Full table | Only unclustered files |
| Data skew handling | Poor | Poor | Built-in |
| Multi-column | No | Yes (~4 cols) | Yes |
| Auto-maintenance | No | No | Yes (via `OPTIMIZE`) |
| Recommended | Legacy | UC tables (manual) | UC managed tables |

---
**Key concepts**

- **`CLUSTER BY` at table creation** — defines which columns to cluster on:
```sql
CREATE TABLE my_table (id INT, event_date DATE, region STRING)
CLUSTER BY (event_date, region);
```

- **Incremental `OPTIMIZE`** — only newly added/unclustered files are processed. On a large table, this means `OPTIMIZE` gets cheaper over time, not more expensive.

- **Changing cluster columns** — unlike partitioning, you can change `CLUSTER BY` columns without recreating the table:
```sql
ALTER TABLE my_table CLUSTER BY (region);
```

- **No data skew** — Liquid Clustering distributes data evenly regardless of value frequency, solving the classic partition skew problem (e.g., one date having 10x more rows than others).

- **Predictive Optimization** — on Serverless / Unity Catalog, Databricks can automatically run `OPTIMIZE` for you in the background.

---
**When to use it**
- UC managed Delta tables (your current setup)
- High-cardinality columns (user_id, transaction_id)
- Query patterns that change over time
- Tables with uneven data distribution

In [0]:
df1 = spark.read.parquet("abfss://dalta-lake-lab-sacc-container@daltalakelabstorageacc.dfs.core.windows.net/invoices/invoices_101_200.parquet")
df2 = spark.read.parquet("abfss://dalta-lake-lab-sacc-container@daltalakelabstorageacc.dfs.core.windows.net/invoices/invoices_1_100.parquet")
df3 = spark.read.parquet("abfss://dalta-lake-lab-sacc-container@daltalakelabstorageacc.dfs.core.windows.net/invoices/invoices_201_99457.parquet")

print("df1 count:", df1.count())
print("df2 count:", df2.count())
print("df3 count:", df3.count())

In [0]:
df_union = df1.union(df2).union(df3)
print("df_union rows count :- ", df_union.count())

**Note :- Please Read This First**

- First we creating a table with **hive style partitioning** and applying **z-order** over it.
- Then will create a table with **liquid clustering**.
- And will **compare the performace of both**.

#### Creating Delta Table With Hive Style Partitioning And Applying ZOrder

In [0]:
df_union.write.mode("overwrite").partitionBy("invoice_date").saveAsTable("delta_catalog.delta_db.lc_tbl_ex1")

In [0]:
%sql
OPTIMIZE delta_catalog.delta_db.lc_tbl_ex1
ZORDER BY customer_id; --In ZORDER By we put columns that we are using on our queries to filter/prune the table.

In [0]:
%sql
SELECT count(*) FROM delta_catalog.delta_db.lc_tbl_ex1;

#### Creating Delta Table With Liquid Clustering.

In [0]:
df_union.write.mode("overwrite").clusterBy("invoice_date", "customer_id").saveAsTable("delta_catalog.delta_db.lc_tbl_ex2")

In [0]:
%sql
SELECT
  count(*)
FROM
  delta_catalog.delta_db.lc_tbl_ex2;

#### Comparing/Testing Both Tables Performance

In [0]:
%%time
spark.sql("""
    SELECT
        category,
        SUM(price * quantity) AS total_sales
    FROM
        delta_catalog.delta_db.lc_tbl_ex1
    WHERE
        customer_id = 201
        AND (invoice_date BETWEEN '2022-01-01' AND '2023-12-31')
    GROUP BY
        category
        """)

In [0]:
%%time
spark.sql("""
    SELECT
        category,
        SUM(price * quantity) AS total_sales
    FROM
        delta_catalog.delta_db.lc_tbl_ex2
    WHERE
        customer_id = 201
        AND (invoice_date BETWEEN '2022-01-01' AND '2023-12-31')
    GROUP BY
        category
        """)